# 02 — Cleaning and Exploratory Data Analysis (EDA)

**Project:** Forecasting Youth Unemployment in Pakistan
**Author:** Awais Ahmed Shah — BS Data Science (Final Year), UET Peshawar

## Goal
Load the processed World Bank dataset built in notebook 01, check its quality
(missing values, ranges, consistency), and explore the main patterns in Pakistan's
youth unemployment, 1991–2025:

- How has youth unemployment changed over time?
- How large is the gap between young women and young men?
- How does youth unemployment compare with overall unemployment?
- How do participation and GDP per capita move alongside it?

All charts are saved to `results/figures/`.

In [1]:
from pathlib import Path

def find_project_root():
    """Walk up from the notebook folder until the folder containing data/ is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (folder containing data/processed).")

ROOT = find_project_root()
print("Project root:", ROOT)

Project root: /home/hatch/workspace/youth-unemployment-repo


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3})

FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "processed" / "pakistan_labour_indicators_1991_2025.csv")
df = df.sort_values("year").reset_index(drop=True)
print("Shape:", df.shape)
df.head()

Shape: (35, 7)


/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


,year,youth_unemployment_total,youth_unemployment_male,youth_unemployment_female,unemployment_total,labour_force_participation_total,gdp_per_capita_usd
0,1991,0.960,1.032,0.577,0.596,49.68,380.019843
1,1992,0.958,1.030,0.573,0.587,49.68,395.677209
2,1993,0.967,1.037,0.594,0.596,49.20,407.888141
3,1994,0.972,1.042,0.604,0.601,49.15,399.955928
4,1995,0.973,1.029,0.611,0.611,48.36,450.550279


## Step 1 — Data-quality checks

In [3]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nSummary statistics:")
df.describe().round(3)

Missing values per column:
year                                0
youth_unemployment_total            0
youth_unemployment_male             0
youth_unemployment_female           0
unemployment_total                  0
labour_force_participation_total    0
gdp_per_capita_usd                  0
dtype: int64

Data types:
year                                  int64
youth_unemployment_total            float64
youth_unemployment_male             float64
youth_unemployment_female           float64
unemployment_total                  float64
labour_force_participation_total    float64
gdp_per_capita_usd                  float64
dtype: object

Summary statistics:


,year,youth_unemployment_total,youth_unemployment_male,youth_unemployment_female,unemployment_total,labour_force_participation_total,gdp_per_capita_usd
count,35.000,35.000,35.000,35.000,35.000,35.000,35.000
mean,2008.000,3.841,3.840,3.758,2.076,50.742,960.947
std,10.247,3.762,3.641,4.358,2.040,1.055,426.195
min,1991.000,0.820,0.901,0.493,0.398,48.360,380.020
25%,1999.500,0.963,1.026,0.597,0.599,50.064,528.467
50%,2008.000,0.990,1.047,0.668,0.620,50.590,987.305
75%,2016.500,7.119,7.486,6.180,3.616,51.519,1370.397
max,2025.000,11.053,9.994,14.278,6.338,52.730,1595.909


In [4]:
# Cleaning decisions (documented, reproducible):
# - Years are already a complete annual sequence 1991-2025 with no gaps.
# - No values were imputed in this project: every indicator has full coverage
#   (verified by the missing-value count above). If a future update introduces
#   gaps, they must be documented here before any imputation is applied.
# - Column names were standardised in notebook 01; here we only sort by year.
clean = df.copy()
clean.to_csv(ROOT / "data" / "processed" / "pakistan_labour_clean.csv", index=False)
print("Saved cleaned dataset: data/processed/pakistan_labour_clean.csv")

Saved cleaned dataset: data/processed/pakistan_labour_clean.csv


## Step 2 — The headline trend: youth unemployment, 1991–2025

In [5]:
youth = clean.set_index("year")["youth_unemployment_total"]
print(f"Lowest : {youth.min():.3f}% in {int(youth.idxmin())}")
print(f"Highest: {youth.max():.3f}% in {int(youth.idxmax())}")
print(f"1991   : {youth.loc[1991]:.3f}%")
print(f"2025   : {youth.loc[2025]:.3f}%")
print(f"Mean 1991-2025: {youth.mean():.3f}%")

fig, ax = plt.subplots()
ax.plot(youth.index, youth.values, marker="o", ms=3, color="#1f77b4")
ax.set_xlabel("Year"); ax.set_ylabel("Youth unemployment (% of labour force 15-24)")
ax.set_title("Pakistan — Youth unemployment rate, 1991–2025 (World Bank / ILO modelled)")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_youth_unemployment_trend.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig1_youth_unemployment_trend.png")

Lowest : 0.820% in 2007
Highest: 11.053% in 2021
1991   : 0.960%
2025   : 9.594%
Mean 1991-2025: 3.841%


Saved results/figures/fig1_youth_unemployment_trend.png


**Reading the chart honestly.** The World Bank series is a *modelled ILO estimate*.
Values in the early 1990s are close to 1%, which is far below every later decade, and the
series climbs steeply around the turn of the century. This pattern should be read with
care: it may partly reflect changes in estimation and labour-force measurement rather
than a pure economic change. The safer claims are about the last two decades, where the
series moves in a narrower, higher band and spikes visibly in 2020–2021 (the COVID-19
period) before easing back.

## Step 3 — Young women vs young men

In [6]:
male = clean.set_index("year")["youth_unemployment_male"]
female = clean.set_index("year")["youth_unemployment_female"]
gap_2025 = female.loc[2025] - male.loc[2025]
print(f"2025 male  : {male.loc[2025]:.3f}%")
print(f"2025 female: {female.loc[2025]:.3f}%")
print(f"2025 gender gap (female - male): {gap_2025:.3f} percentage points")
print(f"Mean gap 1991-2025: {(female - male).mean():.3f} pp; "
      f"years where female > male: {(female > male).sum()} of {len(female)}")

fig, ax = plt.subplots()
ax.plot(male.index, male.values, marker="o", ms=3, label="Youth male")
ax.plot(female.index, female.values, marker="s", ms=3, label="Youth female")
ax.set_xlabel("Year"); ax.set_ylabel("Youth unemployment (%)")
ax.set_title("Pakistan — Youth unemployment by gender, 1991–2025")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_youth_by_gender.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig2_youth_by_gender.png")

2025 male  : 9.416%
2025 female: 10.143%
2025 gender gap (female - male): 0.727 percentage points
Mean gap 1991-2025: -0.083 pp; years where female > male: 8 of 35


Saved results/figures/fig2_youth_by_gender.png


## Step 4 — Youth vs overall unemployment

In [7]:
total = clean.set_index("year")["unemployment_total"]
print(f"2025 youth unemployment : {youth.loc[2025]:.3f}%")
print(f"2025 total unemployment : {total.loc[2025]:.3f}%")
print(f"2025 ratio youth/total  : {youth.loc[2025] / total.loc[2025]:.2f}x")

fig, ax = plt.subplots()
ax.plot(youth.index, youth.values, marker="o", ms=3, label="Youth (15-24)")
ax.plot(total.index, total.values, marker="^", ms=3, label="Total (all ages)")
ax.set_xlabel("Year"); ax.set_ylabel("Unemployment (%)")
ax.set_title("Pakistan — Youth vs total unemployment, 1991–2025")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_youth_vs_total.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig3_youth_vs_total.png")

2025 youth unemployment : 9.594%
2025 total unemployment : 5.420%
2025 ratio youth/total  : 1.77x


Saved results/figures/fig3_youth_vs_total.png


## Step 5 — Participation and GDP per capita (context indicators)

In [8]:
part = clean.set_index("year")["labour_force_participation_total"]
gdp = clean.set_index("year")["gdp_per_capita_usd"]
print("Correlations with youth unemployment (Pearson, full period):")
print(f"  youth vs labour-force participation: {youth.corr(part):.3f}")
print(f"  youth vs GDP per capita (current US$): {youth.corr(gdp):.3f}")
print("Note: correlation over a strongly trending period is suggestive only,",
      "not evidence of causation.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(part.index, part.values, marker="o", ms=3, color="#2ca02c")
axes[0].set_title("Labour force participation, total (%)"); axes[0].set_xlabel("Year")
axes[1].plot(gdp.index, gdp.values, marker="o", ms=3, color="#9467bd")
axes[1].set_title("GDP per capita (current US$)"); axes[1].set_xlabel("Year")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig4_participation_and_gdp.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig4_participation_and_gdp.png")

Correlations with youth unemployment (Pearson, full period):
  youth vs labour-force participation: 0.845
  youth vs GDP per capita (current US$): 0.838
Note: correlation over a strongly trending period is suggestive only, not evidence of causation.


Saved results/figures/fig4_participation_and_gdp.png


## Key findings (from the numbers printed above)
1. Youth unemployment in 2025 stands at **9.594%** in the World Bank modelled series —
   about **1.8x** the total unemployment rate for the same year.
2. The gender pattern changes over time: young women's unemployment exceeds young men's
   in only 8 of the 35 years — but those years are 2008, 2015 and every year from 2020 to
   2025. In 2025 the gap is +0.73 percentage points (female 10.143% vs male 9.416%), and
   the highest female values in the series occur in 2020–2021.
3. The series peaks in **2021** in this dataset, coinciding with the COVID-19 shock period,
   then declines in 2022–2023 and edges up again in 2024–2025.
4. Early-1990s values are near 1% and are not directly comparable in spirit with recent
   decades — a limitation carried into the modelling in notebook 03 and the final report.